# Ditto Client

> The core abstraction for ditto client.

In [ ]:
#| default_exp clients.ditto

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## ClientDitto

This class implements the basic abstraction of a client and is considered the base class for all type of clients

In [ ]:
#| export
@AlgorithmRegistry.register_client("ditto")
class ClientDitto(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        

### Training

In [ ]:
#| export
def ptrain(self: ClientDitto):
    self.model = self.model.to(self.device)
    self.model_per = self.model_per.to(self.device)
    self.model.train()
    self.model_per.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            self.optimizer.zero_grad()
            X, y = batch[self.data_key], batch[self.label_key]
            outputs = self.model_per(X)
            loss = self.criterion(outputs, y)
            self.optimizer_per.zero_grad()
            loss.backward()
            self.optimizer_per.step(self.model.parameters(), self.device)


In [ ]:
#| export
def train(self: ClientDitto):
    self.model = self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            output = self.model(X)
            loss = self.criterion(output, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

In [ ]:
#| export
def fit(self: ClientDitto):
    self.ptrain()
    self.train()

### Saving

In [ ]:
#| export
@patch
def save_state(self: ClientDitto, save_to_disk= False):

    client_state = {
        "model": self.model.state_dict(),
        "model_per": self.model_per.state_dict(),
        'optimizer': self.optimizer.state_dict(),
        "optimizer_per": self.optimizer_per.state_dict()
    }

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))
        torch.save(client_state, state_path)

    return client_state
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()